In [2]:
import json

dataset = json.load(open("fine-tune_dataset.json","r"))
print(len(dataset))
print(dataset[1])

7
{'input': 'Je bent een uiterst kritische AI Recruiter. Je doel is een eerlijke vergelijking te maken tussen het CV en de Vacature en een motivatiebrief te schrijven volgens de VDAB-normen.\r\n\r\n\r\n\r\n    ---\r\n\r\n    VACATURE:\r\n\r\n    Benefits\r\n\r\nPulled from the full job description\r\n\r\nCompany car\r\n\r\nFood allowance\r\n\r\nEco vouchers\r\n\r\n&nbsp;\r\n\r\nFull job description\r\n\r\nCompany Description\r\n\r\n\r\n\r\nSopra Steria offers tailored, end-to-end corporate technology and software solutions to help clients make bold choices and deliver results. Successfully so! With more than 51.000 colleagues in nearly 30 countries, we rank as Europe’s leading digital solutions provider. Some of the most successful companies in Europe rely on our technology due to our commitment to innovation, collaboration, and value in business development.\r\n\r\n\r\n\r\nThe world is how we shape it. Let’s shape it together.\r\n\r\n\r\n\r\n\r\n\r\nJob Description\r\n\r\n\r\n\r\nAs a

In [3]:
!pip install unsloth trl peft accelerate bitsandbytes
!pip install unsloth_zoo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 113.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224

In [4]:
# For GPU check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

CUDA available: True
GPU: Tesla T4


In [5]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

max_seq_length = 4096  # Verhoogd voor lange prompts (vacature + CV + motivatiebrief)
dtype = None

# Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-7b-instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [6]:
from datasets import Dataset

def format_prompt(item):
    """Format training data using the model's native chat template"""
    messages = [
        {"role": "user", "content": item['input']},
        {"role": "assistant", "content": json.dumps(item['output'], ensure_ascii=False)}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

formatted_data = [format_prompt(item) for item in dataset]
print(formatted_data[0][:500])  # Print first 500 chars to verify format
dataset = Dataset.from_dict({"text": formatted_data})
print(dataset)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Je bent een uiterst kritische AI Recruiter. Je doel is een eerlijke vergelijking te maken tussen het CV en de Vacature en een motivatiebrief te schrijven volgens de VDAB-normen.

---
VACATURE:
Job opening: Frontend developer
Peliqan is a highly scalable and secure cloud solution for data collaboration in the modern data stack. We are on a mission to reinvent how data is shared in co
Dataset({
    features: ['text'],
    num_rows: 7
})


In [7]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16, # Verlaagd naar 16: stabieler voor kleinere datasets
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32, # Altijd 2x de rank (16 * 2)
    lora_dropout=0,  # Supports any, but = 0 is optimized
    bias="none",     # Supports any, but = "none" is optimized
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized version
    random_state=3407,
    use_rslora=False,  # Rank stabilized LoRA
    loftq_config=None, # LoftQ
)

Unsloth 2026.3.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Training arguments optimized for Unsloth
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,  # Verlaagd voor kleine dataset
        warmup_steps=5,
        num_train_epochs=3,  # 3 epochs in plaats van max_steps
        learning_rate=1e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        save_strategy="epoch",
        save_total_limit=2,
        dataloader_pin_memory=False,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/7 [00:00<?, ? examples/s]

In [9]:
# Train the model
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7 | Num Epochs = 3 | Total steps = 21
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 1 x 1) = 1
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Step,Training Loss
1,2.493602
2,2.622785
3,2.378091
4,2.152602
5,2.661355
6,2.056456
7,4.438522
8,1.977863
9,2.205897
10,2.062524


In [10]:
# Test the fine-tuned model
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

test_prompt = """Je bent een uiterst kritische AI Recruiter. Je doel is een eerlijke vergelijking te maken tussen het CV en de Vacature en een motivatiebrief te schrijven volgens de VDAB-normen.

---
VACATURE:
Functieomschrijving
Hoe zal jouw takenpakket eruit zien?

Je werkt dagelijks aan complexe en innovatieve applicaties in Java en Angular.
Je gaat te werk in een Agile team, samen met ervaren analisten en ontwikkelaars, dompel je je onder in het functionele en technische design, testing en documentatie.
Je staat in voor de technische implementaties en integraties, waar je betrokken zal worden in architecturale keuzes en kan rekenen op een goede ondersteuning van een Solution Architect.

Profiel
Wat verwachten zij van jou?

Je beschikt over enkele jaren ervaring als java developer, we mikken op een minimum van 5 jaar.
Java, Angular en Spring Boot zijn je dan ook niet onbekend. Ervaring in Jira en database management zijn alvast pluspunten!
Met je hands-on mentaliteit ben je een echte teamplayer en leer je snel bij!

Professionele vaardigheden
Technische ICT-ontwikkelingen documenteren
UX/UI-interfaces ontwerpen
De vraag van de klant analyseren
ICT-applicaties ontwikkelen
Testprocedures voor informaticaprogramma's en -applicaties opstellen
Technische specificaties opstellen
De ontwikkelde toepassingen testen
Een front-end ontwikkelen
Programmeren in een specifieke computertaal

Aanbod
Vergoeding: min. 4000 - max. 4500 euro bruto per maand

Plaats tewerkstelling
8000 BRUGGE

---
CV:
Lauren Nijman Hoofdstraat 12, Amsterdam lauren-nijman@example.com PROFIEL: Ervaren resultaatgerichte software engineer met BSc Informatica en vier jaar ervaring. Verbeterde applicatieperformance met 30%. WERKERVARING: - Software ontwikkelaar bij Tech Solutions B.V. (jan 2023 - heden): CI/CD-pijplijnen, migratie monolith naar microservices. - Software engineer bij Innovatieve Software Oplossingen B.V. (jan 2021 - jan 2023): Microservices architectuur, AWS migratie. VAARDIGHEDEN: Java programmeren, Software ontwikkelen, Data structuren, Software testen, Algoritme ontwerp, AWS, CI/CD.

---

STRIKTE ANALYSE REGELS:
1. MATCHED SKILLS: Alleen vaardigheden die LETTERLIJK of als DIRECT SYNONIEM in beide teksten voorkomen.
2. MISSING SKILLS: Alleen concrete vereisten uit de vacature die ontbreken op het CV.
3. MATCH PERCENTAGE:
   - 80-100%: Perfecte match.
   - 60-79%: Goede basis, mist details.
   - 40-59%: Verschillend vakgebied, maar overdraagbare technische skills.
   - <40%: Geen relevante match.

RICHTLIJNEN VOOR DE VDAB-MOTIVATIEBRIEF:
    Schrijf een volledige brief in het veld 'motivation_letter' met deze opbouw:
    1. CONTACTGEGEVENS KANDIDAAT
    2. CONTACTGEGEVENS BEDRIJF
    3. ONDERWERP: Gebruik de exacte functietitel uit de vacature.
    4. AANSPREKING
    5. INLEIDING
    6. MOTIVATIE & TROEVEN
    7. AFSLUITING
    8. GROET: Eindig met "Met vriendelijke groet," gevolgd door de volledige naam.

STRIKTE VOORWAARDEN:
- Gebruik NOOIT placeholders (zoals [Naam]).
- Gebruik \\n voor alle witregels en alinea-overgangen in de JSON string.
- Schrijf in professioneel, foutloos Nederlands (u-vorm)

GEEF UITSLUITEND JSON TERUG:
{
    "match_percentage": <int>,
    "matched_skills": ["skill1", "skill2"],
    "missing_skills": ["skill1", "skill2"],
    "motivation_letter": "...",
    "match_text": "[VOLLEDIGE NAAM VAN CV] heeft een [PERFECTE | GOEDE | GEMIDDELDE | LAGE] match met de vacature."
}"""

# Inference - alleen user message, GEEN assistant prefill
messages = [
    {"role": "user", "content": test_prompt},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# Generate response
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=2048,
    use_cache=True,
    temperature=0.1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

# Decode and print
response = tokenizer.batch_decode(outputs)[0]
print(response)

# Extract JSON from response (between ```json and ``` or the last {...} block)
import re

# Methode 1: Zoek JSON in markdown code block
code_block_match = re.search(r'```json\s*(.+?)\s*```', response, re.DOTALL)
if code_block_match:
    json_str = code_block_match.group(1).strip()
    print("\n\n=== EXTRACTED JSON (from code block) ===")
    print(json_str)
    try:
        parsed = json.loads(json_str)
        print("\n=== PARSED SUCCESSFULLY ===")
        print(json.dumps(parsed, indent=2, ensure_ascii=False))
    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
else:
    # Methode 2: Zoek de laatste { ... } die "match_percentage" bevat
    # Gebruik een greedy search voor geneste braces
    all_json_candidates = re.findall(r'\{[\s\S]*?"match_percentage"[\s\S]*?\}\s*', response, re.MULTILINE)
    if all_json_candidates:
        print("\n\n=== EXTRACTED JSON (fallback) ===")
        print(all_json_candidates[-1])

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Je bent een uiterst kritische AI Recruiter. Je doel is een eerlijke vergelijking te maken tussen het CV en de Vacature en een motivatiebrief te schrijven volgens de VDAB-normen.

---
VACATURE:
Functieomschrijving
Hoe zal jouw takenpakket eruit zien?

Je werkt dagelijks aan complexe en innovatieve applicaties in Java en Angular.
Je gaat te werk in een Agile team, samen met ervaren analisten en ontwikkelaars, dompel je je onder in het functionele en technische design, testing en documentatie.
Je staat in voor de technische implementaties en integraties, waar je betrokken zal worden in architecturale keuzes en kan rekenen op een goede ondersteuning van een Solution Architect.

Profiel
Wat verwachten zij van jou?

Je beschikt over enkele jaren ervaring als java developer, we mikken op een minimum van 5 jaar.
Java, Angular en Spring Boot zijn je dan ook niet onbekend. Ervaring 

In [11]:
# model.save_pretrained_gguf("gguf_model", tokenizer, quantization_method="q4_k_m")

from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
folder_name = f"open-jobs_gguf_model_{timestamp}"

model.save_pretrained_gguf(folder_name, tokenizer, quantization_method="q4_k_m")


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [06:09<00:00, 92.44s/it]


Unsloth: Merge process complete. Saved to `/content/open-jobs_gguf_model_20260311_155331`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['open-jobs_gguf_model_20260311_155331_gguf/Qwen2.5-7B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['open-jobs_gguf_model_20260311_155331_gguf/Qwen2.5-7B-Instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model open-jobs_gguf_model_20260311_155331_gguf/Qwen2.5-7B-Instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to open-jobs_gguf_model_20260311_155331_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f open-jobs_gguf_model_20260311_155331_gguf/Modelfile


{'save_directory': 'open-jobs_gguf_model_20260311_155331',
 'gguf_directory': 'open-jobs_gguf_model_20260311_155331_gguf',
 'gguf_files': ['open-jobs_gguf_model_20260311_155331_gguf/Qwen2.5-7B-Instruct.Q4_K_M.gguf'],
 'modelfile_location': 'open-jobs_gguf_model_20260311_155331_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [ ]:
%pip install huggingface-hub

In [2]:
import os
print(f"Current directory: {os.getcwd()}")
from os.path import join, dirname
from dotenv import load_dotenv

dotenv_path = join(dirname("../.."), '.env')
load_dotenv(dotenv_path)

hugging_face = os.getenv('open_job')
print(f"Hugging Face API Key: {hugging_face[:4]}...{hugging_face[-4:]}")  # Print first and last 4 chars for verification

Current directory: /Users/buraq-mac/Documents/GitHub/OpenJob/oj_backend/fine-tune


TypeError: 'NoneType' object is not subscriptable

In [ ]:
from huggingface_hub import login

load_dotenv()
hugging_face = os.getenv('open_job')

login(hugging_face)

# In Unsloth gebruik je daarna dit om te uploaden voor Ollama:
model.push_to_hub_gguf(
    "Vibe-Bataklik/open_jobs",
    tokenizer,
    quantization_method = "q4_k_m", # Aanbevolen voor Ollama
    token = hugging_face,
)

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...5-7B-Instruct.Q4_K_M.gguf:   0%|          | 15.9MB / 4.68GB            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/Vibe-Bataklik/open_jobs
Unsloth: Cleaning up temporary files...


'Vibe-Bataklik/open_jobs'

In [ ]:
import json
import re
from datetime import datetime

# ── Performance Evaluation ────────────────────────────────────────────────────
# Evalueer het model op de laatste 10 entries van de dataset en sla op per model.

def get_bucket(pct):
    if pct >= 80: return "PERFECTE"
    if pct >= 60: return "GOEDE"
    if pct >= 40: return "GEMIDDELDE"
    return "LAGE"

def extract_json_safe(text):
    m = re.search(r'```json\s*(.*?)\s*```', text, re.DOTALL)
    if m:
        try: return json.loads(m.group(1))
        except: pass
    start, end = text.rfind('{'), text.rfind('}') + 1
    if start != -1 and end > start:
        try: return json.loads(text[start:end])
        except: pass
    return None

# Laad dataset en gebruik de laatste 10 entries als eval-set
eval_dataset = json.load(open("fine-tune_dataset.json", "r"))
eval_sample = eval_dataset[-10:]

valid_json_count = 0
pct_errors = []
bucket_correct = 0
details = []

FastLanguageModel.for_inference(model)
print(f"Evaluating {len(eval_sample)} samples...\n")

for i, item in enumerate(eval_sample):
    expected_pct = item['output']['match_percentage']
    expected_bucket = get_bucket(expected_pct)

    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": item['input']}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        out = model.generate(
            input_ids=inputs, max_new_tokens=1024, use_cache=True,
            temperature=0.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    response_text = tokenizer.decode(out[0], skip_special_tokens=True)
    parsed = extract_json_safe(response_text)

    row = {
        "sample_index": len(eval_dataset) - len(eval_sample) + i,
        "expected_pct": expected_pct,
        "expected_bucket": expected_bucket,
        "valid_json": False,
        "predicted_pct": None,
        "predicted_bucket": None,
        "pct_error": None,
        "bucket_correct": False,
    }

    if parsed and "match_percentage" in parsed:
        valid_json_count += 1
        pred_pct = parsed["match_percentage"]
        pred_bucket = get_bucket(pred_pct)
        error = abs(pred_pct - expected_pct)
        row.update({
            "valid_json": True,
            "predicted_pct": pred_pct,
            "predicted_bucket": pred_bucket,
            "pct_error": error,
            "bucket_correct": pred_bucket == expected_bucket,
        })
        pct_errors.append(error)
        if pred_bucket == expected_bucket:
            bucket_correct += 1

    details.append(row)
    status = f"{pred_pct}% ({pred_bucket})" if row['valid_json'] else "INVALID JSON"
    print(f"  [{i+1:2d}] expected={expected_pct:3d}% ({expected_bucket:10s}) | predicted={status}")

n = len(eval_sample)
metrics = {
    "model_name": folder_name,
    "evaluated_at": datetime.now().isoformat(),
    "num_samples": n,
    "valid_json_rate": round(valid_json_count / n, 3),
    "mean_absolute_error_pct": round(sum(pct_errors) / len(pct_errors), 1) if pct_errors else None,
    "bucket_accuracy": round(bucket_correct / n, 3),
    "details": details,
}

perf_file = f"performance_{folder_name}.json"
with open(perf_file, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print(f"\n=== Performance Summary: {folder_name} ===")
print(f"  Valid JSON rate  : {metrics['valid_json_rate']*100:.0f}%")
print(f"  Mean abs error   : {metrics['mean_absolute_error_pct']}% match-verschil")
print(f"  Bucket accuracy  : {metrics['bucket_accuracy']*100:.0f}% correct (LAGE/GEMIDDELDE/GOEDE/PERFECTE)")
print(f"  Opgeslagen in    : {perf_file}")
